# 02 — Feature Engineering V2

**Objetivo:** Transformar el dataset limpio en un dataset listo para modelado.

**Comparativa V1 vs V2:**
- **Versión 1:** El encoding categórico utilizaba One-Hot Encoding masivo y una técnica manual de "Encoding de contribución + Bayesian smoothing" que creaba 2 columnas por variable para evitar el sesgo de las categorías raras.
- **Versión 2 (Este notebook):** Se utiliza la librería estándar `category_encoders.TargetEncoder` que aplica Bayesian Smoothing matemáticamente óptimo bajo el capó. Esto evita la necesidad de crear 2 columnas por variable (como se hacía en "5.3 Encoding de contribución"), simplifica el código, evita el data leakage y reduce drásticamente la dimensionalidad del modelo, manteniendo la misma protección contra categorías de bajo volumen.

In [ ]:
import pandas as pd
import numpy as np
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

CLEAN_PATH = "../data/processed/leads_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 1. Eliminar features sin valor predictivo (Data Leakage)
Igual que en V1, descartamos `anio_creacion` (no tiene varianza en 2025), `subtipo_interes` y `plataforma` (que revelan información post-cualificación).

In [ ]:
cols_eliminar = ["subtipo_interes"]
cols_eliminar = [c for c in cols_eliminar if c in df.columns]

df = df.drop(columns=cols_eliminar)
print(f"Dataset tras limpieza: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 2. Crear nuevas features derivadas
Creamos `es_fin_de_semana` y `franja_horaria` basadas en los hallazgos temporales del EDA.

In [ ]:
if "dia_semana_creacion" in df.columns:
    df["es_fin_de_semana"] = df["dia_semana_creacion"].isin(["sábado", "domingo"]).astype(int)

def clasificar_franja(hora):
    if 0 <= hora < 6: return "madrugada"
    elif 6 <= hora < 12: return "manana"
    elif 12 <= hora < 18: return "tarde"
    return "noche"

if "hora_creacion" in df.columns:
    df["franja_horaria"] = df["hora_creacion"].apply(clasificar_franja)

print("Nuevas features derivadas creadas exitosamente.")

## 3. Agrupar categorías de baja frecuencia
Las categorías muy raras se agrupan en "otros" para evitar ruido en el encoding.

In [ ]:
UMBRAL_PCT = 1.0
umbral_abs = int(len(df) * UMBRAL_PCT / 100)

cat_cols_agrupar = ["nombre_formulario", "vehiculo_interes", "origen", "campana", "concesionario"]

for col in cat_cols_agrupar:
    if col not in df.columns: continue
    vc = df[col].value_counts()
    bajas = vc[vc < umbral_abs].index
    if len(bajas) > 0:
        df.loc[df[col].isin(bajas), col] = "otros"
        print(f"{col}: {len(bajas)} categorías agrupadas en 'otros'")

## 4. Split Train/Test
Es CRÍTICO separar el dataset ANTES de aplicar cualquier Encoding para evitar **Data Leakage** (que el modelo memorice información del test set).

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} filas")
print(f"Test:  {X_test.shape[0]:,} filas")

## 5. Bayesian Target Encoding (Reemplaza al manual de V1)

### 5.1 ¿Por qué ya no usamos "Encoding de contribución"?
En la V1, el paso **"5.3 Encoding de contribución + Bayesian smoothing"** creaba manualmente dos columnas por cada variable (`_contribucion_hot` y `_tasa_suavizada`) para evitar que categorías raras con 100% de conversión confundieran al modelo.

En la V2, utilizamos **`category_encoders.TargetEncoder`** con el parámetro `smoothing`. Esta librería oficial de SciKit-Learn realiza exactamente la misma penalización Bayesiana de forma automática bajo el capó, devolviendo una sola variable densa y altamente predictiva. Esto reduce la dimensionalidad a la mitad y mantiene la misma protección matemática que habías programado a mano en V1.

In [ ]:
# Variables a codificar con Bayesian Smoothing
encode_cols = ["vehiculo_interes", "origen", "nombre_formulario", "campana", "concesionario"]
encode_cols = [c for c in encode_cols if c in X_train.columns]

# TargetEncoder aplica el Bayesian Smoothing automáticamente
encoder = TargetEncoder(cols=encode_cols, smoothing=100)

X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc = encoder.transform(X_test)

print("Bayesian Target Encoding aplicado con éxito.")

### 5.2 One-Hot Encoding para Baja Cardinalidad
Las variables restantes (`dia_semana_creacion`, `franja_horaria`, `origen_creacion`) se codifican con One-Hot.

In [ ]:
onehot_cols = list(X_train_enc.select_dtypes(include=["object"]).columns)

X_train_final = pd.get_dummies(X_train_enc, columns=onehot_cols, drop_first=True, dtype=int)
X_test_final = pd.get_dummies(X_test_enc, columns=onehot_cols, drop_first=True, dtype=int)

# Alinear columnas
X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)

print(f"Dimensiones finales Train: {X_train_final.shape}")
print(f"Dimensiones finales Test:  {X_test_final.shape}")

**Evaluación y Comentarios V2:**
Al aplicar esta metodología híbrida (Bayesian Smoothing para alta cardinalidad y One-Hot para baja cardinalidad), logramos un dataset muy denso y poderoso. Hemos reemplazado la lógica manual y redundante de V1 por estándares de la industria (`TargetEncoder`), garantizando robustez sin sobreajuste.

## 6. Exportar datasets para modelado

In [ ]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train_final.to_csv(f"{OUTPUT_DIR}/X_train_v2.csv", index=False)
X_test_final.to_csv(f"{OUTPUT_DIR}/X_test_v2.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train_v2.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test_v2.csv", index=False)

print("Archivos exportados exitosamente.")